# Dự đoán bệnh tiểu đường (Pima Indians Diabetes)

## Định nghĩa bài toán
Dữ liệu của Viện Quốc gia về Bệnh tiểu đường và Rối loạn tiêu hóa - Nhằm mục đích dự đoán dựa trên các phép đo chẩn đoán xem một bệnh nhân có mắc bệnh tiểu đường hay không.

- **Input:** 8 chỉ số sức khỏe của bệnh nhân gồm:
  1. `Pregnancies`: Số lần mang thai
  2. `Glucose`: Nồng độ Glucose huyết tương
  3. `BloodPressure`: Huyết áp tâm trương (mm Hg)
  4. `SkinThickness`: Độ dày nếp gấp da cơ đầu (mm)
  5. `Insulin`: Insulin huyết thanh trong 2 giờ (mu U/ml)
  6. `BMI`: Chỉ số khối cơ thể (cân nặng/chiều cao^2)
  7. `DiabetesPedigreeFunction`: Chức năng phả hệ bệnh tiểu đường
  8. `Age`: Tuổi (năm)
- **Output:** `Outcome`: Biến mục tiêu có giá trị (0: không bệnh, 1: có bệnh di truyền / tiểu đường).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

## 1. Đọc dữ liệu
Data được đọc từ thư mục `data/pima-indians-diabetes.csv`. File không có header.

In [ ]:
# Khai báo tên các cột dựa trên mô tả dữ liệu Pima Indians
column_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
                'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Tiến hành load dữ liệu
df = pd.read_csv('data/pima-indians-diabetes.csv', header=None, names=column_names)

display(df.head())
print(f"\nKích thước tập dữ liệu: {df.shape}")

## 2. Khám phá dữ liệu (EDA) và Cảnh báo Outliers
Với dữ liệu y tế, có những thông số tuyệt đối **không thể bằng 0** (Ví dụ như Huyết áp, BMI, Mức Glucose). Ta sẽ phát hiện các ngoại lai (Outliers) mang giá trị 0 phi lý này bằng cách vẽ Boxplot.

In [ ]:
# 2.1. Vẽ ma trận độ tương quan (Heatmap) để tìm hiểu tính chất các loại bệnh
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='RdYlBu', fmt='.2f', linewidths=0.5)
plt.title("Ma trận tương quan các chỉ số y tế")
plt.show()

In [ ]:
# 2.2. Vẽ Boxplot cho Glucose, BloodPressure, BMI để xác minh ngoại lai giá trị 0
features_to_check = ['Glucose', 'BloodPressure', 'BMI']

plt.figure(figsize=(12, 6))
sns.boxplot(data=df[features_to_check], orient="h", palette="pastel")
plt.title("Boxplot cho Glucose, BloodPressure và BMI - Chứng minh tồn tại giá trị 0 vô lý")
plt.show()

# Nhận xét: Nhìn vào Boxplot, ta thấy rõ các điểm dị biệt (Outliers) tụt thẳng xuống mức 0 ở trục hoành. 
# Huyết áp hay Glucose bằng 0 ở người sống là điều bất khả thi, nên đây là dữ liệu bị thiếu hoặc ghi nhận lỗi.

## 3. Xử lý Outliers & Dữ liệu bị rỗng
Chúng ta sẽ thay các giá trị 0 ở các cột nhạy cảm bằng `NaN`, sau đó sử dụng thuật toán `SimpleImputer` để thay thế thông minh (điền chiến lược trung vị - `median`). _Tuyệt đối không dùng dropna() vì sẽ làm mất mát lượng lớn dữ liệu quý giá của y tế._

In [ ]:
# Thay thế các giá trị 0 vô lý thành NaN ở các cột cần thiết
print("Số lượng giá trị 0 vô lý trước khi xử lý:")
for col in features_to_check:
    print(f"{col}: {(df[col] == 0).sum()} dòng")

df[features_to_check] = df[features_to_check].replace(0, np.nan)

print("\nĐã thay giá trị 0 bằng NaN. Tiến hành SimpleImputer...")

# Tách X, y
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Dùng SimpleImputer(strategy='median') điền các giá trị NaN
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# Đưa trở lại thành DataFrame
X = pd.DataFrame(X_imputed, columns=X.columns)

## 4. Tiền xử lý, Chia Dataset 70/30 và Chuẩn Hóa
Dữ liệu y tế thang đo rất chênh lệch (Ví dụ Insulin đo hàng trăm trong khi BMI là hàng chục). Việc quy chuẩn (`StandardScaler`) là bắt buộc!

In [ ]:
# Cắt 30% cho tập Test, random_state=42 để kết quả tái diễn
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Sử dụng StandardScaler để Scale dữ liệu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scale thành công! Kích thước data Train:", X_train_scaled.shape)

## 5. Huấn luyện (Modeling) & Đánh Giá
Sử dụng 2 mô hình là `RandomForestClassifier` (sức mạnh cây quyết định phân loại) và `SVC` để tiên lượng bệnh. So sánh dựa vào đặc điểm của `F1-Score` (chỉ số quan trọng trong y tế vì nó kết hợp cả Precision và Recall).

In [ ]:
# --- 1. Huấn luyện Random Forest ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_y_pred = rf_model.predict(X_test_scaled)

# --- 2. Huấn luyện SVC ---
svc_model = SVC(random_state=42)
svc_model.fit(X_train_scaled, y_train)
svc_y_pred = svc_model.predict(X_test_scaled)

In [ ]:
# Hàm in Confusion Matrix chuyên dụng
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='OrRd', 
                xticklabels=['Không bệnh (0)', 'Bệnh (1)'], 
                yticklabels=['Không bệnh (0)', 'Bệnh (1)'])
    plt.title(title)
    plt.ylabel('Thực tế')
    plt.xlabel('Dự đoán')
    plt.show()

print("=== MA TRẬN NHẦM LẪN (CONFUSION MATRIX) ===")
plot_cm(y_test, rf_y_pred, "Confusion Matrix - Random Forest")
plot_cm(y_test, svc_y_pred, "Confusion Matrix - SVC")

### Nhận xét & Kết quả (So sánh 2 thuật toán)

In [ ]:
print("=== CHỈ SỐ ĐÁNH GIÁ TRỌNG ĐIỂM Y TẾ ===\n")

rf_f1 = f1_score(y_test, rf_y_pred)
svc_f1 = f1_score(y_test, svc_y_pred)

print("1. Random Forest Classifier:")
print(f"   - Accuracy: {accuracy_score(y_test, rf_y_pred):.4f}")
print(f"   - F1-Score: {rf_f1:.4f}\n")

print("2. Support Vector Classifier (SVC):")
print(f"   - Accuracy: {accuracy_score(y_test, svc_y_pred):.4f}")
print(f"   - F1-Score: {svc_f1:.4f}\n")

# Logic in nhận xét tự động
if rf_f1 > svc_f1:
    print("-> NHẬN XÉT: Dựa trên F1-Score, thuật toán **Random Forest** đang cho kết quả bắt bệnh toàn diện tốt hơn khi cân bằng giữa độ chính xác dự đoán số người bị bệnh và số người khỏe thực. Trong y tế thì điểm F1 rất quan trọng vì nó trừng phạt các mô hình bắt nhầm nhiều.")
elif svc_f1 > rf_f1:
    print("-> NHẬN XÉT: Dựa trên F1-Score, thuật toán **SVC** đang cho kết quả bắt bệnh toàn diện tốt hơn khi cân bằng giữa độ chính xác dự đoán số người bệnh thực tế. Cả đường bao ranh giới Support Vector Margin khá dẻo dai thích hợp cho bộ data này!")
else:
    print("-> NHẬN XÉT: Cả hai thuật toán đều cho khả năng bắt bệnh xuất sắc tương đương nhau!")
